# Naive Bayes Classifier

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from time import time

In [ ]:
df = pd.read_csv("../datasets/airlines_delay.csv", sep=",")
AirlineUnique = df.Airline.unique()
AirportFromUnique = df.AirportFrom.unique()
AirportToUnique = df.AirportTo.unique()
Airlinelst = list(range(len(AirlineUnique)))
df['NumAirline'] = df['Airline']
df['NumAirline'].replace(AirlineUnique, Airlinelst, inplace=True)
AirportFromlst = list(range(len(AirportFromUnique)))
df['NumAirportFrom'] = df['AirportFrom']
df['NumAirportFrom'].replace(AirportFromUnique, AirportFromlst, inplace=True)
AirportTolst = list(range(len(AirportToUnique)))
df['NumAirportTo'] = df['AirportTo']
df['NumAirportTo'].replace(AirportToUnique, AirportTolst, inplace=True)

df = df.sample(n=10000)
X = df[["Length","NumAirline","NumAirportFrom","NumAirportTo","DayOfWeek"]]
y = df['Class']


/var/folders/61/4f_9vd3x7c9_5dsr15qd4fww0000gn/T/ipykernel_38005/3641257505.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['NumAirline'].replace(AirlineUnique, Airlinelst, inplace=True)
/var/folders/61/4f_9vd3x7c9_5dsr15qd4fww0000gn/T/ipykernel_38005/3641257505.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_d

In [3]:
X_simulated_small, y_simulated_small = make_classification(n_samples=300, n_features=6, n_classes=2, random_state=1)
X_simulated_large, y_simulated_large = make_classification(n_samples=15000, n_features=6, n_classes=2, random_state=1)

In [4]:
def max_class(x, prior_list, mean_list, variance_list):
    likelihoods=[]
    post=[]
    for idx in range(2):
        likelihood_num = np.exp((-1/2)*((x-mean_list[idx])**2)/(2*variance_list[idx]))
        likelihood_den = np.sqrt(2*np.pi*variance_list[idx])
        likelihoods.append(likelihood_num/likelihood_den)
    post.append(np.log(prior_list[0]) + np.sum(np.log(likelihoods[0])))
    post.append(np.log(prior_list[1]) + np.sum(np.log(likelihoods[1])))
    return np.argmax(post)

from multiprocessing.pool import ThreadPool

def Parallel_NB(X_train, X_test, y_train):
    n=len(X_train)
    m=X_train.shape[1]
    prior_list=np.zeros(2, dtype=float)
    mean_list=np.zeros((2,m), dtype=float)
    variance_list=np.zeros((2,m), dtype=float)
    for index in range(2):
        sub_df = X_train[y_train==index]
        prior_list[index] = len(sub_df)/n
        mean_list[index,:] = sub_df.mean(axis=0)
        variance_list[index,:] = sub_df.var(axis=0)
    X_testing = X_test.values
    Xis=[row.tolist() for row in X_testing]
    pool = ThreadPool(5)
    y_pred = [pool.apply(max_class, args=(Xi, prior_list, mean_list, variance_list)) for Xi in Xis]
    return y_pred

def test_accuracy(true, pred):
    correct=sum(a==b for a,b in zip(true,pred))
    return correct/len(true)


In [5]:
# Simulated Study Naive Bayes (small data)
start=time()
acc=[]
for i in range(10):
    X_train,X_test,y_train,y_test=train_test_split(X_simulated_small,y_simulated_small,test_size=0.2,random_state=i)
    X_train=pd.DataFrame(X_train)
    X_test=pd.DataFrame(X_test)
    y_train=pd.Series(y_train)
    pred=Parallel_NB(X_train,X_test,y_train)
    acc.append(test_accuracy(y_test,pred))
print('Prediction accuracy of model:',sum(acc)/len(acc))
print('Training time for Naive Bayes:',time()-start)


Prediction accuracy of model: 0.9133333333333334
Training time for Naive Bayes: 0.11201095581054688


In [6]:
# Simulated Study Naive Bayes (large data)
start=time()
acc=[]
for i in range(10):
    X_train,X_test,y_train,y_test=train_test_split(X_simulated_large,y_simulated_large,test_size=0.2,random_state=i)
    X_train=pd.DataFrame(X_train)
    X_test=pd.DataFrame(X_test)
    y_train=pd.Series(y_train)
    pred=Parallel_NB(X_train,X_test,y_train)
    acc.append(test_accuracy(y_test,pred))
print('Prediction accuracy of model:',sum(acc)/len(acc))
print('Training time for Naive Bayes:',time()-start)


Prediction accuracy of model: 0.8930000000000001
Training time for Naive Bayes: 3.0133349895477295


In [7]:
# Real data study for Naive Bayes
start=time()
acc=[]
for i in range(10):
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=i,shuffle=True)
    pred=Parallel_NB(X_train,X_test,y_train)
    acc.append(test_accuracy(y_test.values,pred))
print('Prediction accuracy of model:',sum(acc)/len(acc))
print('Training time for Naive Bayes:',time()-start)


Prediction accuracy of model: 0.56455
Training time for Naive Bayes: 1.9758367538452148
